# 4. Connected floor planning: keep connected items close

Lecture: *Week 11 - Floor Planning*, from the *Connected 2 Dimensional Packing* slide to *Results for different floor spaces* (slides 24–29).

**The problem.** The four tables of the *Floor Planning* slide, $(w, h) = (4,5), (3,7), (6,4), (7,7)$, must be placed on a fixed $W \times H$ floor without overlapping, and each may be turned through 90°. Some tables are connected (think of machines joined by pipes): $t_1$–$t_2$, $t_1$–$t_3$, $t_1$–$t_4$, $t_2$–$t_3$ and $t_3$–$t_4$, as drawn on the *Clustering of components* slide. Place them so that the total **rectilinear distance** between the centres of connected tables (across plus up/down, like walking along city blocks) is as small as possible.

**How to run it.** Run the cells in order, starting with the first: it installs Pyomo and the HiGHS solver (allow a minute the first time). With only four tables the model is small.

**Expected result.** `Status: optimal`, the smallest total distance for the 20 × 20 floor, the position of each table, and a plot with the five connections drawn between the table centres. Several layouts can be optimal, so positions and rotations can differ between runs and solvers and from the pictures on the slide; the total distance is the same for every optimal layout.

*Written for this repository from the formulas on slides 26–28. The MATLAB code behind the pictures on the Results slide is not in the course files.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

## Load the libraries

Pyomo builds and solves the model, NumPy holds the data and Matplotlib draws the layout.

In [ ]:
from pyomo.environ import *
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## The data

The four tables, the floor size and the connected pairs. The floor is 20 × 20, the first picture on the *Results* slide (see *Try it* for the other two). `M` switches constraints off, as in the other notebooks; no switched-off constraint can need more than $W + H$ plus the longest side, so that is large enough.

In [ ]:
# Data: width w[i] and height h[i] of tables 1-4 (Floor Planning slide)
w = np.array([4,3,6,7])
h = np.array([5,7,4,7])
W, H = 20, 20      # floor size (first picture on the Results slide)

# connected pairs (i, j) with i < j, from the Clustering of components slide
connected = [(1,2), (1,3), (1,4), (2,3), (3,4)]

M = W + H + max(w.max(), h.max())   # big-M: large enough to switch a constraint off
no_items = len(w)

## The model

The variables of notebook 1 without the square side `v` (the floor is fixed), plus two distance variables for each connected pair:

- `x[i]`, `y[i]`: the bottom-left corner of table $i$;
- `xij[i, j]`, `yij[i, j]`: the two binaries of each pair, which choose the one non-overlap constraint that must hold;
- `r[i]`: 1 if table $i$ is rotated, 0 if not;
- `R[i, j]`, `B[i, j]`: the horizontal and vertical distance between the centres of a connected pair ($R_{ij}$ and $B_{ij}$ on the slides).

In [ ]:
# Model
model = ConcreteModel()
model.I = RangeSet(no_items)
model.J = RangeSet(no_items)
model.C = Set(initialize=connected, dimen=2)          # the connected pairs
model.w = Param(model.I, initialize=lambda model, i: int(w[i-1]))
model.h = Param(model.I, initialize=lambda model, i: int(h[i-1]))

model.x = Var(model.I, within=NonNegativeReals)       # bottom-left corner of table i
model.y = Var(model.I, within=NonNegativeReals)
model.xij = Var(model.I, model.J, within=Binary)      # which side of each other two tables are
model.yij = Var(model.I, model.J, within=Binary)
model.r = Var(model.I, within=Binary)                 # 1 = table i rotated by 90 degrees
model.R = Var(model.C, within=NonNegativeReals)       # horizontal distance between the centres of i and j
model.B = Var(model.C, within=NonNegativeReals)       # vertical distance between the centres of i and j

After rotation, table $i$ is $(1-r_i)\,w_i + r_i\,h_i$ wide and $(1-r_i)\,h_i + r_i\,w_i$ high, so its centre is half of that from its corner. $D^x_{ij}$ and $D^y_{ij}$ are the differences between the centres of tables $i$ and $j$ (slide *Distance between items*); they can be negative.

In [ ]:
def width(model, i):    # width of table i after rotation
    return (1 - model.r[i]) * model.w[i] + model.r[i] * model.h[i]

def height(model, i):   # height of table i after rotation
    return (1 - model.r[i]) * model.h[i] + model.r[i] * model.w[i]

def Dx(model, i, j):    # centre of i minus centre of j, horizontally
    return (model.x[i] + width(model, i) / 2) - (model.x[j] + width(model, j) / 2)

def Dy(model, i, j):    # centre of i minus centre of j, vertically
    return (model.y[i] + height(model, i) / 2) - (model.y[j] + height(model, j) / 2)

## Objective

Minimise the total rectilinear distance $\sum_{(i,j)} (R_{ij} + B_{ij})$ over the connected pairs (slide *Objective Function*).

In [ ]:
# Objective: total rectilinear distance between connected tables
model.obj = Objective(expr=sum(model.R[i,j] + model.B[i,j] for (i,j) in model.C), sense=minimize)

## Constraints

Three groups, as on the slides:

1. **Boundary constraints:** every table lies inside the $W \times H$ floor (slide *Amending the constraints*).
2. **Non-overlap constraints:** the four constraints of notebook 1 for each pair $i < j$; $(x_{ij}, y_{ij})$ chooses the one that must hold and $M$ switches the other three off.
3. **Absolute distance:** $R_{ij} \ge D^x_{ij}$ and $R_{ij} \ge -D^x_{ij}$, so $R_{ij} \ge |D^x_{ij}|$; the same for $B_{ij}$. The model only asks for "at least", but because we minimise, the solver makes $R_{ij}$ and $B_{ij}$ exactly equal to the distances.

In [ ]:
# 1. boundary constraints: every table inside the W x H floor
model.box_xi = Constraint(model.I, rule=lambda model, i: model.x[i] + width(model, i) <= W)
model.box_yi = Constraint(model.I, rule=lambda model, i: model.y[i] + height(model, i) <= H)

# 2. non-overlap: the other three choices add M or 2M and are switched off
# (xij, yij) = (0, 0): table i is LEFT of table j
def overlap_xij_rule(model, i, j):
    if i < j:
        return model.x[i] + width(model, i) <= model.x[j] + M*(model.xij[i,j] + model.yij[i,j])
    return Constraint.Skip
model.overlap_xij = Constraint(model.I, model.J, rule=overlap_xij_rule)

# (xij, yij) = (1, 0): table j is LEFT of table i
def overlap_xji_rule(model, i, j):
    if i < j:
        return model.x[j] + width(model, j) <= model.x[i] + M*(1 - model.xij[i,j] + model.yij[i,j])
    return Constraint.Skip
model.overlap_xji = Constraint(model.I, model.J, rule=overlap_xji_rule)

# (xij, yij) = (0, 1): table i is BELOW table j
def overlap_yij_rule(model, i, j):
    if i < j:
        return model.y[i] + height(model, i) <= model.y[j] + M*(1 + model.xij[i,j] - model.yij[i,j])
    return Constraint.Skip
model.overlap_yij = Constraint(model.I, model.J, rule=overlap_yij_rule)

# (xij, yij) = (1, 1): table j is BELOW table i
def overlap_yji_rule(model, i, j):
    if i < j:
        return model.y[j] + height(model, j) <= model.y[i] + M*(2 - model.xij[i,j] - model.yij[i,j])
    return Constraint.Skip
model.overlap_yji = Constraint(model.I, model.J, rule=overlap_yji_rule)

# 3. absolute distance: R >= |Dx| and B >= |Dy| (minimising makes them equal)
model.R_pos = Constraint(model.C, rule=lambda model, i, j:  Dx(model, i, j) <= model.R[i,j])
model.R_neg = Constraint(model.C, rule=lambda model, i, j: -Dx(model, i, j) <= model.R[i,j])
model.B_pos = Constraint(model.C, rule=lambda model, i, j:  Dy(model, i, j) <= model.B[i,j])
model.B_neg = Constraint(model.C, rule=lambda model, i, j: -Dy(model, i, j) <= model.B[i,j])

## Solve

`tee=False` hides the solver log; the cell prints the status (`optimal` means proved best) and the total distance.

In [ ]:
# Solve
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)
print("Status:", results.solver.termination_condition)
print("Total rectilinear distance:", round(value(model.obj), 3))

## Results

List each table's corner and rotation and the distances of each connected pair, then draw the layout inside the dashed floor outline. Each connection is drawn as a rectilinear path between the two centres: across, then up or down. Binary values can come back as numbers such as 0.9999999, so the code tests `> 0.5`. A value printed as `-0` is 0.

In [ ]:
# Size after rotation and centre of each table (a table is rotated when r[i] is 1; test > 0.5)
size, cx, cy = {}, {}, {}
for i in model.I:
    rot = model.r[i].value > 0.5
    size[i] = (h[i-1], w[i-1]) if rot else (w[i-1], h[i-1])
    cx[i] = model.x[i].value + size[i][0] / 2
    cy[i] = model.y[i].value + size[i][1] / 2
    print(f"table {i}: {w[i-1]}x{h[i-1]}{' rotated' if rot else ''} at ({model.x[i].value:g}, {model.y[i].value:g})")
for (i, j) in connected:
    print(f"t{i}-t{j}: across {abs(cx[i] - cx[j]):g} + up/down {abs(cy[i] - cy[j]):g}")

fig, ax = plt.subplots()
cmap = plt.get_cmap('Pastel1')
for i in model.I:
    ax.add_patch(patches.Rectangle((model.x[i].value, model.y[i].value), size[i][0], size[i][1], linewidth=1.5,
                                   edgecolor='black', facecolor=cmap(i / no_items), label=str(i)))
for (i, j) in connected:   # rectilinear path between the centres
    ax.plot([cx[i], cx[j], cx[j]], [cy[i], cy[i], cy[j]], linewidth=2)
ax.add_patch(patches.Rectangle((0, 0), W, H, fill=False, linestyle='--', edgecolor='grey'))
ax.set_xlim(-1, W + 1); ax.set_ylim(-1, H + 1); ax.set_aspect('equal')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.grid(True); ax.legend(loc='upper right')
plt.show()

## What do you notice?

However the tables are placed, the total distance can never be 0. Why not?

<details><summary>Answer</summary>

The distance is measured between the **centres**. Two tables that do not overlap are separated left/right or up/down, so their centres are at least half of one width (or height) plus half of the other apart in that direction. For example, if $t_1$ (4 wide) sits directly beside $t_2$ (3 wide), their centres are at least $2 + 1.5 = 3.5$ apart. The solver can only make these gaps as small as the sizes allow.
</details>

## Try it

Change the floor to `W, H = 8, 20` and then to `W, H = 10, 15` (the other two pictures on the *Results* slide) and choose *Run → Run All Cells* each time. Can the total distance be smaller than on the 20 × 20 floor?

<details><summary>Answer</summary>

No. Every layout that fits on an 8 × 20 or a 10 × 15 floor also fits on the 20 × 20 floor, so a smaller floor can only remove layouts, never add better ones: the total distance stays the same or grows. The middle picture on the slide shows how the narrow 8 × 20 floor pushes the tables into a column.
</details>